<a href="https://colab.research.google.com/github/lfbedoyamichaelpage-maker/colab-automation-test/blob/main/test_automation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import json
import io
from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaIoBaseUpload

# 1. AUTENTICACIÓN
if os.path.exists('secret_key.json'):
    with open('secret_key.json') as f:
        info = json.load(f)
    creds = service_account.Credentials.from_service_account_info(info)
    print("✅ Autenticado desde GitHub")
else:
    from google.colab import auth
    auth.authenticate_user()
    import google.auth
    creds, _ = google.auth.default()

service = build('drive', 'v3', credentials=creds)

# 2. LEER EL ARCHIVO 'datos.txt'
query = "name = 'datos.txt' and trashed = false"
results = service.files().list(q=query, fields="files(id, name, parents)").execute()
items = results.get('files', [])

if not items:
    print("❌ No se encontró el archivo datos.txt")
else:
    file_id = items[0]['id']
    parent_id = items[0]['parents'][0] # Guardaremos la salida en la misma carpeta

    # Descargar contenido
    request = service.files().get_media(fileId=file_id)
    fh = io.BytesIO()
    downloader = MediaIoBaseDownload(fh, request)
    done = False
    while not done:
        _, done = downloader.next_chunk()

    texto_original = fh.getvalue().decode('utf-8')
    print(f"📖 Leído de Drive: {texto_original}")

    # 3. PROCESAR Y ESCRIBIR SALIDA
    nuevo_contenido = f"{texto_original}\n--- Procesado por GitHub Actions el {os.popen('date').read()} ---"

    file_metadata = {
        'name': 'resultado_automatizacion.txt',
        'parents': [parent_id]
    }
    media = MediaIoBaseUpload(io.BytesIO(nuevo_contenido.encode('utf-8')),
                              mimetype='text/plain',
                              resumable=True)

    # Crear el nuevo archivo en Drive
    file_salida = service.files().create(body=file_metadata,
                                        media_body=media,
                                        fields='id').execute()

    print(f"✅ ¡Éxito! Archivo de salida creado con ID: {file_salida.get('id')}")